# 03c — NHS GP Surgery Locations Audit

**Purpose:** Audit GP surgery locations, geocode via Code-Point Open, validate spatial
distribution, compute GP density per 100,000 population, and cross-reference with IMD
deprivation.

**Input:** `data/raw/poi/epraccur.csv` — downloaded via NHS ORD 2-0-0 API (RO177,
England, Status=Active)
**Expected:** ~12,214 rows, 4 columns (OrgId, Name, PostCode, LastChangeDate)
**Dependencies:** `data/audit/postcode_lookup.parquet`, `data/audit/master_lsoa_table.parquet`
**Output:** `data/audit/gp_surgeries_geocoded.parquet`

In [1]:
import re

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from loguru import logger
from pathlib import Path
from pyproj import Transformer

ROOT = Path("/Users/souravamseekarmarti/Projects/aequitas")
GP_PATH = ROOT / "data/raw/poi/epraccur.csv"
POSTCODE_LOOKUP_PATH = ROOT / "data/audit/postcode_lookup.parquet"
MASTER_LSOA_PATH = ROOT / "data/audit/master_lsoa_table.parquet"
OUTPUT_PATH = ROOT / "data/audit/gp_surgeries_geocoded.parquet"
FIG_PATH = ROOT / "data/audit/fig_03c_gp_surgeries.png"

checks: list[tuple[str, bool, str]] = []

## 1. Load and Profile epraccur.csv

In [2]:
# The file has a header row: OrgId, Name, PostCode, LastChangeDate
df_raw = pd.read_csv(GP_PATH)
logger.info(f"Loaded epraccur.csv with header: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols")
logger.info(f"Columns detected: {df_raw.columns.tolist()}")

# Standardise column names regardless of how the file was provided
df_raw.columns = ["OrgId", "Name", "PostCode", "LastChangeDate"]
df = df_raw.copy()

logger.info(f"Assigned columns: {df.columns.tolist()}")
logger.info(f"Final shape: {df.shape[0]:,} rows x {df.shape[1]} cols")

print("=== Column inventory ===")
for col in df.columns:
    n_null = df[col].isna().sum()
    dtype = df[col].dtype
    if dtype == object:
        n_unique = df[col].nunique()
        sample = df[col].dropna().iloc[0] if n_null < len(df) else "N/A"
        print(f"  {col!r}: dtype={dtype}, nulls={n_null}, unique={n_unique}, sample={sample!r}")
    else:
        print(f"  {col!r}: dtype={dtype}, nulls={n_null}, min={df[col].min()}, max={df[col].max()}")

print("\n=== Head ===")
print(df.head(5).to_string())

2026-03-13 09:38:12.073 | INFO     | __main__:<module>:3 - Loaded epraccur.csv with header: 12213 rows x 4 cols


2026-03-13 09:38:12.074 | INFO     | __main__:<module>:4 - Columns detected: ['OrgId', 'Name', 'PostCode', 'LastChangeDate']


2026-03-13 09:38:12.074 | INFO     | __main__:<module>:10 - Assigned columns: ['OrgId', 'Name', 'PostCode', 'LastChangeDate']


2026-03-13 09:38:12.074 | INFO     | __main__:<module>:11 - Final shape: 12,213 rows x 4 cols


=== Column inventory ===
  'OrgId': dtype=str, nulls=0, min=A81002, max=Y09021
  'Name': dtype=str, nulls=0, min=(FRACTURE CLINIC) NORTH OOH, max=ZETLAND MEDICAL PRACTICE
  'PostCode': dtype=str, nulls=0, min=AL1 3FY, max=YO8 9BX
  'LastChangeDate': dtype=str, nulls=0, min=2014-04-15, max=2026-03-11

=== Head ===
    OrgId                             Name  PostCode LastChangeDate
0  A81002       QUEENS PARK MEDICAL CENTRE  TS18 2AW     2024-06-11
1  A81004            ACKLAM MEDICAL CENTRE   TS5 8SB     2023-08-22
2  A81005               SPRINGWOOD SURGERY  TS14 7DJ     2023-08-22
3  A81006  TENNANT STREET MEDICAL PRACTICE  TS18 2AT     2024-08-31
4  A81007                BANKHOUSE SURGERY  TS24 7PW     2023-08-22


## 2. Validate: row count, OrgId uniqueness, postcode format

In [3]:
# Row count — expect ~12,214
expected_rows = 12_214
row_count_ok = abs(len(df) - expected_rows) <= 10  # allow ±10 for minor API variation
checks.append((f"Row count ~{expected_rows:,}", row_count_ok, f"Got {len(df):,}"))
logger.info(f"Row count: {len(df):,} -- {'PASS' if row_count_ok else 'FAIL'}")

# OrgId uniqueness
n_unique_orgid = df["OrgId"].nunique()
orgid_unique_ok = n_unique_orgid == len(df)
checks.append(("OrgId all unique", orgid_unique_ok, f"Unique={n_unique_orgid}, Total={len(df)}"))
logger.info(f"OrgId uniqueness: {n_unique_orgid}/{len(df)} -- {'PASS' if orgid_unique_ok else 'FAIL'}")

if not orgid_unique_ok:
    dupes = df[df.duplicated("OrgId", keep=False)].sort_values("OrgId")
    print(f"\nDuplicate OrgIds ({len(dupes)} rows):")
    print(dupes.head(20).to_string())

# Postcode format: [A-Z]{1,2}[0-9][0-9A-Z]? [0-9][A-Z]{2}
PC_REGEX = re.compile(r"^[A-Z]{1,2}[0-9][0-9A-Z]? [0-9][A-Z]{2}$")
n_null_pc = df["PostCode"].isna().sum()
valid_pc = df["PostCode"].dropna().apply(lambda x: bool(PC_REGEX.match(str(x).strip().upper())))
n_valid_pc = valid_pc.sum()
n_total_nonnull = len(valid_pc)
pc_format_ok = n_valid_pc == n_total_nonnull and n_null_pc == 0
checks.append((
    "All postcodes match standard format (zero nulls)",
    pc_format_ok,
    f"Valid={n_valid_pc}/{n_total_nonnull}, Nulls={n_null_pc}",
))
logger.info(f"Postcode format: {n_valid_pc}/{n_total_nonnull} valid, {n_null_pc} nulls -- {'PASS' if pc_format_ok else 'FAIL'}")

if not pc_format_ok:
    bad_mask = ~valid_pc.reindex(df.index, fill_value=False)
    bad = df[bad_mask | df["PostCode"].isna()]
    print("\nInvalid/null postcodes:")
    print(bad[["OrgId", "Name", "PostCode"]].head(20).to_string())

2026-03-13 09:38:12.084 | INFO     | __main__:<module>:5 - Row count: 12,213 -- PASS


2026-03-13 09:38:12.085 | INFO     | __main__:<module>:11 - OrgId uniqueness: 12213/12213 -- PASS


2026-03-13 09:38:12.089 | INFO     | __main__:<module>:30 - Postcode format: 12213/12213 valid, 0 nulls -- PASS


## 3. Geocode via Code-Point Open postcode lookup

In [4]:
def standardise_postcode(pc: str) -> str | None:
    """Standardise postcode to format 'OUTWARD INWARD' with single space."""
    if pd.isna(pc):
        return None
    pc = re.sub(r"\s+", "", str(pc).strip().upper())
    return pc[:-3] + " " + pc[-3:] if len(pc) > 3 else pc


postcode_lookup = pd.read_parquet(POSTCODE_LOOKUP_PATH)
logger.info(f"Loaded postcode lookup: {len(postcode_lookup):,} postcodes")

df["postcode_std"] = df["PostCode"].apply(standardise_postcode)

# Merge on standardised postcode
merged = df.merge(postcode_lookup, left_on="postcode_std", right_on="Postcode", how="left")
n_matched = merged["Easting"].notna().sum()
match_rate = n_matched / len(merged) * 100
match_ok = match_rate > 95.0
checks.append(("Postcode match rate > 95%", match_ok, f"Matched={n_matched}/{len(merged)} ({match_rate:.2f}%)"))
logger.info(f"Postcode match rate: {match_rate:.2f}% ({n_matched}/{len(merged)}) -- {'PASS' if match_ok else 'FAIL'}")

# Show unmatched postcodes
unmatched = merged[merged["Easting"].isna()][["OrgId", "Name", "PostCode", "postcode_std"]]
if len(unmatched) > 0:
    logger.warning(f"{len(unmatched)} unmatched postcodes:")
    print(unmatched.head(30).to_string())
    if len(unmatched) > 30:
        print(f"  ... and {len(unmatched) - 30} more")
else:
    logger.info("All postcodes matched -- zero unmatched")

2026-03-13 09:38:12.182 | INFO     | __main__:<module>:10 - Loaded postcode lookup: 1,492,016 postcodes


2026-03-13 09:38:12.364 | INFO     | __main__:<module>:20 - Postcode match rate: 98.74% (12059/12213) -- PASS


2026-03-13 09:38:12.365 | WARNING  | __main__:<module>:25 - 154 unmatched postcodes:


       OrgId                                      Name  PostCode postcode_std
256   A85619                       TRAVELLING FAMILIES   NE9 5UR      NE9 5UR
382   A91034     HEADLEY MEDICAL REHABILITATION CENTRE  KT18 6JW     KT18 6JW
444   A91115               KNELLER HALL MEDICAL CENTRE   TW2 7DU      TW2 7DU
490   A91183                  BIELEFELD MEDICAL CENTRE   BF1 0AP      BF1 0AP
491   A91186                            MEDICAL CENTRE   BF1 0DN      BF1 0DN
492   A91192                  PADERBORN MEDICAL CENTRE   BF1 0AF      BF1 0AF
493   A91193                 SENNELAGER MEDICAL CENTRE   BF1 0AB      BF1 0AB
494   A91194                       MONS MEDICAL CENTRE   BF1 2AG      BF1 2AG
495   A91195                            RRU GUHTERSLOH   BF1 0AS      BF1 0AS
496   A91197                   BRUNSSUM MEDICAL CENTRE   BF1 2AH      BF1 2AH
497   A91198                  RAMMSTEIN MEDICAL CENTRE   BF1 0DL      BF1 0DL
498   A91199                     NAPLES MEDICAL CENTRE   BF1 2AB

## 4. Convert Easting/Northing (BNG EPSG:27700) → Lat/Lon (WGS84 EPSG:4326)

In [5]:
transformer = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=False)

mask = merged["Easting"].notna()
lats, lons = transformer.transform(
    merged.loc[mask, "Easting"].values,
    merged.loc[mask, "Northing"].values,
)
merged.loc[mask, "lat"] = lats
merged.loc[mask, "lon"] = lons

logger.info(f"Converted {mask.sum():,} rows Easting/Northing -> lat/lon")
print(f"\nLat range: {merged['lat'].min():.4f} -- {merged['lat'].max():.4f}")
print(f"Lon range: {merged['lon'].min():.4f} -- {merged['lon'].max():.4f}")

2026-03-13 09:38:12.411 | INFO     | __main__:<module>:11 - Converted 12,059 rows Easting/Northing -> lat/lon



Lat range: 49.9126 -- 55.5960
Lon range: -6.3089 -- 1.7551


## 5. Spatial validation: England bounding box, scatter plot, regional distribution

In [6]:
ENG_LAT_MIN, ENG_LAT_MAX = 49.8, 55.9
ENG_LON_MIN, ENG_LON_MAX = -6.5, 1.8

geocoded = merged[merged["lat"].notna()].copy()

lat_ok_mask = geocoded["lat"].between(ENG_LAT_MIN, ENG_LAT_MAX)
lon_ok_mask = geocoded["lon"].between(ENG_LON_MIN, ENG_LON_MAX)
n_outside_bounds = (~(lat_ok_mask & lon_ok_mask)).sum()

bounds_ok = n_outside_bounds == 0
checks.append((
    "All geocoded GP surgeries within England bounding box",
    bounds_ok,
    f"Outside bounds: {n_outside_bounds}",
))
logger.info(f"Spatial bounds check: {n_outside_bounds} outside England bbox -- {'PASS' if bounds_ok else 'FAIL'}")

if n_outside_bounds > 0:
    print("\nOut-of-bounds GP surgeries:")
    print(geocoded[~(lat_ok_mask & lon_ok_mask)][["OrgId", "Name", "PostCode", "lat", "lon"]].to_string())

# Extract postcode area
geocoded = geocoded.copy()
geocoded["pc_area"] = geocoded["PostCode"].str.extract(r"^([A-Z]{1,2})")
merged["pc_area"] = merged["PostCode"].str.extract(r"^([A-Z]{1,2})")

# --- Scatter plot + Regional distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Scatter map
axes[0].scatter(geocoded["lon"], geocoded["lat"], s=2, alpha=0.35, color="#059669", linewidths=0)
axes[0].set_title(f"NHS GP Surgery Locations — England (n={len(geocoded):,})", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].set_xlim(ENG_LON_MIN - 0.2, ENG_LON_MAX + 0.2)
axes[0].set_ylim(ENG_LAT_MIN - 0.2, ENG_LAT_MAX + 0.2)
axes[0].set_aspect("equal")
axes[0].grid(True, alpha=0.3)

# Regional distribution by postcode area (top 30)
area_counts = geocoded["pc_area"].value_counts().sort_values(ascending=False).head(30)
axes[1].barh(area_counts.index[::-1], area_counts.values[::-1], color="#059669", alpha=0.85)
axes[1].set_title("Top 30 Postcode Areas by GP Surgery Count", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Number of GP surgeries")
axes[1].set_ylabel("Postcode area")
axes[1].tick_params(axis="y", labelsize=8)
axes[1].grid(axis="x", alpha=0.3)

plt.suptitle("03c — NHS GP Surgery Locations Audit", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_PATH, dpi=150, bbox_inches="tight")
plt.close()
logger.info(f"Figure saved: {FIG_PATH}")

print(f"\nFull postcode area distribution (top 30 of {geocoded['pc_area'].nunique()} total areas):")
print(area_counts.to_string())

2026-03-13 09:38:12.420 | INFO     | __main__:<module>:16 - Spatial bounds check: 0 outside England bbox -- PASS


2026-03-13 09:38:12.676 | INFO     | __main__:<module>:53 - Figure saved: /Users/souravamseekarmarti/Projects/aequitas/data/audit/fig_03c_gp_surgeries.png



Full postcode area distribution (top 30 of 98 total areas):
pc_area
B     535
M     312
S     312
L     305
CV    262
NE    256
LE    249
DN    236
E     233
SW    218
SE    217
NG    215
PL    199
GL    195
TN    184
LS    184
WA    182
BN    180
N     178
ST    168
BS    165
TS    163
PE    159
BD    152
CM    150
W     150
SS    149
PO    148
RG    148
ME    138


## 6. GP density analysis: GPs per 100,000 population

In [7]:
import glob as _glob

master = pd.read_parquet(MASTER_LSOA_PATH)
logger.info(f"Loaded master_lsoa_table: {len(master):,} LSOAs")

total_population = master["population"].sum()
total_gp_surgeries = len(merged)
total_geocoded = len(geocoded)

national_gp_per_100k = total_geocoded / total_population * 100_000
logger.info(f"Total England population (from master): {total_population:,}")
logger.info(f"Total GP surgeries (all): {total_gp_surgeries:,}")
logger.info(f"Total geocoded GP surgeries: {total_geocoded:,}")
logger.info(f"National GP density: {national_gp_per_100k:.2f} per 100,000 population")

print(f"\n=== GP Density Analysis ===")
print(f"  Total England population:       {total_population:,}")
print(f"  Total GP surgeries (raw):       {total_gp_surgeries:,}")
print(f"  Geocoded GP surgeries:          {total_geocoded:,}")
print(f"  National GPs per 100k pop:      {national_gp_per_100k:.2f}")

# --- Build pc_area population via Code-Point Open postcode → LAD mapping ---
# Code-Point Open: col 0 = postcode, col 8 = LAD code (E-prefixed = England)
_cpo_files = sorted(_glob.glob(str(ROOT / "data/raw/boundaries/code_point_open/Data/CSV/*.csv")))
_dfs = []
for _f in _cpo_files:
    _d = pd.read_csv(_f, header=None, usecols=[0, 8], dtype={0: str, 8: str})
    _dfs.append(_d)
cpo = pd.concat(_dfs, ignore_index=True)
cpo.columns = ["postcode", "lad_cd"]
cpo = cpo[cpo["lad_cd"].notna() & cpo["lad_cd"].str.startswith("E")].copy()
cpo["postcode"] = cpo["postcode"].str.strip()
cpo["pc_area"] = cpo["postcode"].str.extract(r"^([A-Z]{1,2})")
logger.info(f"Code-Point Open England postcodes loaded: {len(cpo):,}")

# For each pc_area, find the dominant LAD code (mode), then aggregate population from master
pc_lad_mode = (
    cpo.groupby("pc_area")["lad_cd"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={"lad_cd": "dominant_lad_cd"})
)

# Sum population per LAD from master
lad_pop = master.groupby("lad_cd")["population"].sum().reset_index().rename(columns={"population": "lad_pop"})

# Build pc_area → approximate population via dominant LAD
pc_pop = pc_lad_mode.merge(lad_pop, left_on="dominant_lad_cd", right_on="lad_cd", how="left")

# GP count per pc_area
area_gp_counts = geocoded["pc_area"].value_counts().reset_index()
area_gp_counts.columns = ["pc_area", "gp_count"]

# Join and compute density
density_df = area_gp_counts.merge(pc_pop[["pc_area", "lad_pop"]], on="pc_area", how="left")
density_df["gp_density_per_100k"] = density_df["gp_count"] / density_df["lad_pop"] * 100_000
density_df = density_df.dropna(subset=["gp_density_per_100k"]).sort_values("gp_density_per_100k", ascending=False)

print(f"\n=== Top 10 Postcode Areas by GP density per 100,000 population ===")
print(density_df[["pc_area", "gp_count", "lad_pop", "gp_density_per_100k"]].head(10).to_string(index=False))

print(f"\n=== Bottom 10 Postcode Areas by GP density per 100,000 population ===")
print(density_df[["pc_area", "gp_count", "lad_pop", "gp_density_per_100k"]].tail(10).to_string(index=False))

# Validate gp_density_per_100k column exists and has non-zero values
density_col_ok = "gp_density_per_100k" in density_df.columns and (density_df["gp_density_per_100k"] > 0).all()
checks.append(("gp_density_per_100k column exists and all non-zero", density_col_ok,
               f"Rows with density: {len(density_df)}, min={density_df['gp_density_per_100k'].min():.2f}, max={density_df['gp_density_per_100k'].max():.2f}"))
logger.info(f"gp_density_per_100k check: {'PASS' if density_col_ok else 'FAIL'}")

2026-03-13 09:38:12.691 | INFO     | __main__:<module>:4 - Loaded master_lsoa_table: 33,755 LSOAs


2026-03-13 09:38:12.691 | INFO     | __main__:<module>:11 - Total England population (from master): 56,490,056


2026-03-13 09:38:12.692 | INFO     | __main__:<module>:12 - Total GP surgeries (all): 12,213


2026-03-13 09:38:12.692 | INFO     | __main__:<module>:13 - Total geocoded GP surgeries: 12,059


2026-03-13 09:38:12.692 | INFO     | __main__:<module>:14 - National GP density: 21.35 per 100,000 population



=== GP Density Analysis ===
  Total England population:       56,490,056
  Total GP surgeries (raw):       12,213
  Geocoded GP surgeries:          12,059
  National GPs per 100k pop:      21.35


2026-03-13 09:38:14.794 | INFO     | __main__:<module>:34 - Code-Point Open England postcodes loaded: 1,491,817


2026-03-13 09:38:14.858 | INFO     | __main__:<module>:69 - gp_density_per_100k check: PASS



=== Top 10 Postcode Areas by GP density per 100,000 population ===
pc_area  gp_count  lad_pop  gp_density_per_100k
     GL       195 121107.0           161.014640
     TN       184 115307.0           159.574007
     NR       129 102974.0           125.274341
     PO       148 140460.0           105.368076
     GU       123 128221.0            95.928124
     RG       148 161431.0            91.680037
     PE       159 180831.0            87.927402
     WA       182 210979.0            86.264510
     BB       133 154737.0            85.952293
     TS       163 196589.0            82.914100

=== Bottom 10 Postcode Areas by GP density per 100,000 population ===
pc_area  gp_count  lad_pop  gp_density_per_100k
     BA        87 571553.0            15.221685
     HD        63 433211.0            14.542567
     DH        71 522081.0            13.599422
     TR        56 570320.0             9.819049
     TA        54 571553.0             9.447943
     HG        55 615489.0             8.9359

## 7. Cross-reference with IMD: GP distribution by deprivation decile

In [8]:
from scipy import stats as _stats

# GP–IMD correlation: do deprived areas have fewer GPs per capita?
# Use postcode-area level: gp_density_per_100k (from Section 6) × mean IMD score
# Mean IMD per pc_area: aggregate master → lad → pc_area via dominant LAD mapping

lad_imd = (
    master.groupby("lad_cd")
    .agg(mean_imd=("imd_score", "mean"))
    .reset_index()
)

# Merge dominant LAD → mean IMD, then link to pc_area
pc_imd = pc_lad_mode.merge(lad_imd, left_on="dominant_lad_cd", right_on="lad_cd", how="left")

# Join onto density_df
corr_df = density_df.merge(pc_imd[["pc_area", "mean_imd"]], on="pc_area", how="inner")
corr_df = corr_df.dropna(subset=["gp_density_per_100k", "mean_imd"])

# Pearson correlation: positive r = more deprived areas have MORE GPs (inverse of expected)
# Expected: negative r (more deprived → fewer GPs per capita)
r, p_value = _stats.pearsonr(corr_df["gp_density_per_100k"], corr_df["mean_imd"])
logger.info(f"GP density vs IMD Pearson r={r:.4f}, p={p_value:.4f} (n={len(corr_df)} postcode areas)")

print(f"\n=== GP–IMD Correlation Analysis ===")
print(f"  Postcode areas in analysis: {len(corr_df)}")
print(f"  Pearson r (GP density per 100k vs mean IMD score): {r:.4f}")
print(f"  p-value: {p_value:.4f}")
if r > 0:
    print("  Interpretation: higher deprivation (higher IMD) correlates with MORE GPs per capita")
    print("  (inverse Marmot effect — urban deprived areas often have more GP provision)")
else:
    print("  Interpretation: higher deprivation (higher IMD) correlates with FEWER GPs per capita")
    print("  (expected 'inverse care law' pattern)")

# Ranking table: postcode area | gp_per_100k | mean_imd | rank
ranking = corr_df[["pc_area", "gp_density_per_100k", "mean_imd"]].sort_values("gp_density_per_100k", ascending=False).reset_index(drop=True)
ranking["rank"] = ranking.index + 1

print(f"\n=== Postcode Area Ranking: GP density vs IMD (top 20) ===")
print(ranking.head(20)[["rank", "pc_area", "gp_density_per_100k", "mean_imd"]].to_string(index=False))

print(f"\n=== Postcode Area Ranking: GP density vs IMD (bottom 10) ===")
print(ranking.tail(10)[["rank", "pc_area", "gp_density_per_100k", "mean_imd"]].to_string(index=False))

# Check: correlation should not be perfect (|r| < 0.9)
imd_corr_ok = abs(r) < 0.9
checks.append((
    "GP–IMD correlation computed and not suspiciously perfect (|r| < 0.9)",
    imd_corr_ok,
    f"r={r:.4f}, p={p_value:.4f}, n={len(corr_df)} postcode areas",
))
logger.info(f"GP-IMD correlation check: {'PASS' if imd_corr_ok else 'FAIL'} (|r|={abs(r):.4f})")

2026-03-13 09:38:15.735 | INFO     | __main__:<module>:23 - GP density vs IMD Pearson r=-0.0586, p=0.5688 (n=97 postcode areas)


2026-03-13 09:38:15.739 | INFO     | __main__:<module>:53 - GP-IMD correlation check: PASS (|r|=0.0586)



=== GP–IMD Correlation Analysis ===
  Postcode areas in analysis: 97
  Pearson r (GP density per 100k vs mean IMD score): -0.0586
  p-value: 0.5688
  Interpretation: higher deprivation (higher IMD) correlates with FEWER GPs per capita
  (expected 'inverse care law' pattern)

=== Postcode Area Ranking: GP density vs IMD (top 20) ===
 rank pc_area  gp_density_per_100k  mean_imd
    1      GL           161.014640 12.721028
    2      TN           159.574007 14.042841
    3      NR           125.274341 21.275113
    4      PO           105.368076 24.514348
    5      GU            95.928124  8.222207
    6      RG            91.680037 11.081707
    7      PE            87.927402 11.718135
    8      WA            86.264510 16.713409
    9      BB            85.952293 36.682912
   10      TS            82.914100 26.255847
   11      KT            82.879299  8.093915
   12      CM            82.639164 12.978965
   13      SS            82.463044 23.136148
   14       N            82.182926 

## 8. Save gp_surgeries_geocoded.parquet — ALL rows retained

In [9]:
output_cols = ["OrgId", "Name", "PostCode", "postcode_std", "LastChangeDate",
               "Easting", "Northing", "lat", "lon", "pc_area"]

# Save ALL rows — unmatched retain NaN lat/lon
output_df = merged[output_cols].copy().reset_index(drop=True)
output_df.to_parquet(OUTPUT_PATH, index=False)
logger.info(f"Saved: {OUTPUT_PATH} -- {len(output_df):,} rows x {len(output_df.columns)} cols")

n_unmatched = output_df["lat"].isna().sum()
n_geocoded_final = len(output_df) - n_unmatched
save_ok = OUTPUT_PATH.exists() and len(output_df) >= 12_200
checks.append(("Output parquet saved successfully (all rows retained)", save_ok, f"Rows: {len(output_df):,}"))
checks.append(("Unmatched GP surgeries retained with NaN coords", True, f"Unmatched (no coords): {n_unmatched}"))
logger.warning(f"{n_unmatched} GP surgeries have no geocoordinates -- retained in parquet with NaN lat/lon")

print(f"\nOutput summary:")
print(f"  Total rows:       {len(output_df):,}")
print(f"  Geocoded:         {n_geocoded_final:,}")
print(f"  Unmatched:        {n_unmatched}")
print(f"  Columns:          {output_df.columns.tolist()}")

2026-03-13 09:38:15.753 | INFO     | __main__:<module>:7 - Saved: /Users/souravamseekarmarti/Projects/aequitas/data/audit/gp_surgeries_geocoded.parquet -- 12,213 rows x 10 cols


2026-03-13 09:38:15.754 | WARNING  | __main__:<module>:14 - 154 GP surgeries have no geocoordinates -- retained in parquet with NaN lat/lon



Output summary:
  Total rows:       12,213
  Geocoded:         12,059
  Unmatched:        154
  Columns:          ['OrgId', 'Name', 'PostCode', 'postcode_std', 'LastChangeDate', 'Easting', 'Northing', 'lat', 'lon', 'pc_area']


## 9. Update figures-registry.md GT-022

In [10]:
FIGURES_REGISTRY = ROOT / "docs/figures-registry.md"
registry_text = FIGURES_REGISTRY.read_text()

new_line = (
    f"| GT-022 | England GP practices (geocoded) | {n_geocoded_final:,} | "
    f"✅ Confirmed | 03c_gp_surgeries.ipynb | "
    f"NHS ODS RO177, England only; no header row in raw file; match rate {match_rate:.1f}% via Code-Point Open |"
)

lines = registry_text.splitlines()
updated_lines = []
registry_ok = False
for line in lines:
    if "GT-022" in line and "GP" in line:
        updated_lines.append(new_line)
        registry_ok = True
        logger.info(f"Updated GT-022 in figures-registry.md: geocoded={n_geocoded_final:,}, match_rate={match_rate:.1f}%")
    else:
        updated_lines.append(line)

if registry_ok:
    FIGURES_REGISTRY.write_text("\n".join(updated_lines) + "\n")
else:
    logger.error("GT-022 not found in figures-registry.md -- manual update required")
    print(f"Replacement line:\n  {new_line}")

checks.append(("figures-registry.md GT-022 updated", registry_ok, f"Geocoded={n_geocoded_final:,}, match rate={match_rate:.1f}%"))

2026-03-13 09:38:15.758 | INFO     | __main__:<module>:17 - Updated GT-022 in figures-registry.md: geocoded=12,059, match_rate=98.7%


## 10. Validation summary

In [11]:
print("\n" + "=" * 65)
print("VALIDATION SUMMARY -- 03c_gp_surgeries")
print("=" * 65)

fail_count = 0
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        fail_count += 1
    print(f"  [{status}] {name}")
    if detail:
        print(f"         -> {detail}")

print("-" * 65)
print(f"  Total checks: {len(checks)}")
print(f"  FAILs:        {fail_count}")
print("=" * 65)

print(f"\nKey figures:")
print(f"  GT-022  Total GP surgeries (raw):   {len(merged):,}")
print(f"  GT-022  Geocoded GP surgeries:       {n_geocoded_final:,}")
print(f"  GT-022  Postcode match rate:         {match_rate:.2f}%")
print(f"  GT-022  National GPs per 100k pop:   {national_gp_per_100k:.2f}")
print(f"  Output: {OUTPUT_PATH}")

assert fail_count == 0, f"{fail_count} FAILs -- see summary above"
logger.success("All checks passed. GP surgeries geocoded dataset saved.")

2026-03-13 09:38:15.763 | SUCCESS  | __main__:<module>:27 - All checks passed. GP surgeries geocoded dataset saved.



VALIDATION SUMMARY -- 03c_gp_surgeries
  [PASS] Row count ~12,214
         -> Got 12,213
  [PASS] OrgId all unique
         -> Unique=12213, Total=12213
  [PASS] All postcodes match standard format (zero nulls)
         -> Valid=12213/12213, Nulls=0
  [PASS] Postcode match rate > 95%
         -> Matched=12059/12213 (98.74%)
  [PASS] All geocoded GP surgeries within England bounding box
         -> Outside bounds: 0
  [PASS] gp_density_per_100k column exists and all non-zero
         -> Rows with density: 97, min=4.16, max=161.01
  [PASS] GP–IMD correlation computed and not suspiciously perfect (|r| < 0.9)
         -> r=-0.0586, p=0.5688, n=97 postcode areas
  [PASS] Output parquet saved successfully (all rows retained)
         -> Rows: 12,213
  [PASS] Unmatched GP surgeries retained with NaN coords
         -> Unmatched (no coords): 154
  [PASS] figures-registry.md GT-022 updated
         -> Geocoded=12,059, match rate=98.7%
-----------------------------------------------------------